# 09 · Idempotent consumers and the commit-before-ack crash

**Level:** advanced · **Time:** 40 minutes · **Prerequisites:** 00, 02, and 07.

Apply a payment-like credit to a local SQLite account, then simulate a crash after committing but before acknowledging. A stable event ID and a database uniqueness constraint stop the repeated delivery from repeating the database side effect.

The deduplication record and the balance change must commit in **the same transaction**. Acknowledgement happens after that commit. This gives one applied database effect per retained event ID in this database; it does not make RabbitMQ or arbitrary external calls exactly-once.

The amounts below are integer demo units, not a financial system.


In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "course" / "rabbit.py").is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from course.rabbit import connect, unique_name, wait_for_message, broker_url
import pika
print("Course root:", ROOT)


## Step 1 — Create an isolated database

The temporary directory belongs only to this run. `processed_events.event_id` is a primary key, making duplicate detection a database operation rather than a race-prone read-then-write check.


In [ ]:
import json
import sqlite3
import tempfile

temporary_directory = tempfile.TemporaryDirectory(prefix="rmqcourse-idempotency-")
database_path = Path(temporary_directory.name) / "consumer.sqlite3"
database = sqlite3.connect(database_path)
database.executescript("""
CREATE TABLE account (id INTEGER PRIMARY KEY, balance INTEGER NOT NULL);
INSERT INTO account VALUES (1, 0);
CREATE TABLE processed_events (event_id TEXT PRIMARY KEY);
""")
database.commit()

def apply_once(event, fail_before_commit=False):
    with database:
        inserted = database.execute(
            "INSERT OR IGNORE INTO processed_events(event_id) VALUES (?)", (event["event_id"],)
        ).rowcount
        if not inserted:
            return False
        database.execute("UPDATE account SET balance = balance + ? WHERE id=1", (event["amount"],))
        if fail_before_commit:
            raise RuntimeError("Simulated failure before database commit")
    return True

def balance():
    return database.execute("SELECT balance FROM account WHERE id=1").fetchone()[0]

assert balance() == 0


## Step 2 — Commit once, then close without acknowledging

Expected balance: 75. Closing the connection with an unacknowledged delivery simulates the delivery consequence of a worker crash. It is deliberate for this demonstration; real code acknowledges after the transaction succeeds.


In [ ]:
queue_name = unique_name("idempotency")
first_event = {"event_id": unique_name("event"), "amount": 75}
with connect() as connection:
    channel = connection.channel()
    channel.queue_declare(queue=queue_name, durable=True)
    channel.confirm_delivery()
    channel.basic_publish(exchange="", routing_key=queue_name,
                          body=json.dumps(first_event).encode(), mandatory=True,
                          properties=pika.BasicProperties(delivery_mode=2, message_id=first_event["event_id"]))
    method, properties, body = wait_for_message(channel, queue_name)
    assert apply_once(json.loads(body)) is True
    print("Committed balance:", balance(), "— deliberately leaving the delivery unacknowledged")
assert balance() == 75


## Step 3 — Receive the duplicate, skip its effect, then acknowledge

Predict that `redelivered` is true, `apply_once` returns false, and the balance stays 75. The same event ID can also arrive in a newly published message whose redelivery flag is false; always deduplicate by application identity, not by that flag alone.


In [ ]:
with connect() as connection:
    channel = connection.channel()
    method, properties, body = wait_for_message(channel, queue_name)
    assert method.redelivered
    applied = apply_once(json.loads(body))
    assert applied is False
    channel.basic_ack(method.delivery_tag)
    print("Redelivered:", method.redelivered, "Applied again:", applied, "Balance:", balance())
assert balance() == 75


## Step 4 — Prove rollback covers both writes

A second event fails before commit. Its balance update and event record must both roll back. We requeue exactly once for this controlled experiment, then process successfully. A deployed worker should use bounded delayed retries such as 08.


In [ ]:
second_event = {"event_id": unique_name("event"), "amount": 25}
with connect() as connection:
    channel = connection.channel()
    channel.confirm_delivery()
    channel.basic_publish(exchange="", routing_key=queue_name,
                          body=json.dumps(second_event).encode(), mandatory=True,
                          properties=pika.BasicProperties(delivery_mode=2, message_id=second_event["event_id"]))
    method, properties, body = wait_for_message(channel, queue_name)
    try:
        apply_once(json.loads(body), fail_before_commit=True)
    except RuntimeError as error:
        print(error)
        channel.basic_nack(method.delivery_tag, requeue=True)
    else:
        raise AssertionError("Expected injected failure")
    assert balance() == 75
    assert database.execute("SELECT COUNT(*) FROM processed_events WHERE event_id=?", (second_event["event_id"],)).fetchone()[0] == 0
    method, properties, body = wait_for_message(channel, queue_name)
    assert apply_once(json.loads(body)) is True
    channel.basic_ack(method.delivery_tag)
assert balance() == 100
print("Final balance:", balance())


## Exercise — Where does this guarantee stop?

Try calling `apply_once(first_event)` three more times. Then explain why inserting the event ID into a database cannot atomically protect a separate HTTP payment request.

**Suggested solution:** all three calls return false and the balance remains 100. For an external service, use its idempotency-key API when available, or persist an outgoing intent and reconcile outcomes. Retain deduplication IDs for at least the maximum replay window. Reusing an ID for a different payload is an application error; deployed systems should detect that, for example by storing a payload hash.

**Troubleshooting:** acknowledgement before commit allows lost effects; recording the ID in a separate transaction can suppress unfinished work. A SQLite file is convenient for a one-process lab; the Django examples use a database transaction and a unique constraint with multiple processes in mind.


In [ ]:
assert [apply_once(first_event) for _ in range(3)] == [False, False, False]
assert balance() == 100
print("Repeated delivery does not repeat the stored effect")


## Cleanup — Remove only this run's queue and temporary database


In [ ]:
with connect() as connection:
    connection.channel().queue_delete(queue=queue_name)
database.close()
temporary_directory.cleanup()
print("Cleaned up")


## References

- [RabbitMQ reliability and redeliveries](https://www.rabbitmq.com/docs/4.2/reliability)
- [Python SQLite transaction control](https://docs.python.org/3/library/sqlite3.html#transaction-control)
